# Deep Dive: Part 2 – Corrective RAG (CRAG)
While Self-RAG focuses on reflective generation during answer construction, Corrective RAG (CRAG) (introduced by Shi-Qi Yan et al., 2024) fixes the problem before generation starts.

In standard RAG, if your vector database returns irrelevant, incomplete, or wrong documents, your generator LLM has no choice but to hallucinate or build a flawed response. CRAG inserts an explicit evaluation and correction layer right after retrieval, classifying documents into strict confidence tiers and triggering dynamic web or database fallback searches when retrieval fails.

## 1. The Core Three-Tiered Action Triad
When documents are fetched from your vector database, a lightweight Retrieval Evaluator assesses the confidence score of the chunks relative to the user query and routes the execution into one of three distinct actions:

In [ ]:
[User Query] ---> [Vector Retrieval] ---> [Retrieval Evaluator]
                                                  ├──> 1. CORRECT   ---> [Knowledge Refinement (Decompose/Filter)]
                                                  ├──> 2. INCORRECT ---> [Discard Chunks & Trigger Web Fallback Search]
                                                  └──> 3. AMBIGUOUS ---> [Combine Refined Chunks + Web Search Extension]

### Action A: Correct (High Confidence)
**Condition:** At least one retrieved document scores above an upper confidence threshold.

**Action:** The system applies a knowledge refinement process (using a decompose-then-recompose algorithm) to strip away irrelevant text snippets, keeping only high-density facts to feed the generator.

### Action B: Incorrect (Low Confidence)
**Condition:** All retrieved documents fall below a lower confidence threshold (the vector database completely failed to find the answer).

**Action:** The system discards all retrieved chunks entirely to prevent noise poisoning. It triggers an external fallback search (such as a large-scale web search via Tavily, Serper, or Google API) to pull fresh external facts.

### Action C: Ambiguous (Moderate Confidence / Mixed Signals)
**Condition:** The retriever returns mixed results—some useful details alongside irrelevant or vague context.

**Action:** The system adopts a hybrid fallback strategy: it refines whatever useful elements exist in the internal documents and augments them with external web search results.

## 2. Implementation Pattern: CRAG Workflow Structure (crag_pipeline.py)
Here is a conceptual implementation of a CRAG workflow using a Python routing pattern:

In [ ]:
"""
crag_pipeline.py
Demonstrates the Corrective RAG (CRAG) evaluation and fallback routing logic.
"""

from typing import List, Dict, Any

class CorrectiveRAGPipeline:
    def __init__(self):
        print("Initializing CRAG Pipeline with Evaluator and Fallback Router...")

    def evaluate_retrieval(self, query: str, retrieved_docs: List[str]) -> str:
        """
        Evaluates retrieved documents against the query.
        Returns one of three action states: 'correct', 'incorrect', or 'ambiguous'.
        """
        # In production, this calls a lightweight LLM or classifier judge
        print(f"Evaluating {len(retrieved_docs)} retrieved chunks for query: '{query}'")
        
        # Simulated evaluation check
        if not retrieved_docs or "error" in retrieved_docs[0].lower():
            return "incorrect"
        elif len(retrieved_docs) == 1 and "partial" in retrieved_docs[0].lower():
            return "ambiguous"
        else:
            return "correct"

    def refine_knowledge_strip(self, docs: List[str]) -> List[str]:
        """Decomposes documents and filters out non-essential noise (Correct action)."""
        print("[CRAG Action: CORRECT] Refining internal knowledge strips...")
        # Strip down sentences to core factual statements
        return [doc.strip() for doc in docs]

    def trigger_web_fallback_search(self, query: str) -> List[str]:
        """Queries external search APIs when internal retrieval fails (Incorrect action)."""
        print(f"[CRAG Action: INCORRECT] Internal retrieval failed. Executing external web search for: '{query}'")
        # Simulated web search fallback result
        return [f"External web search result for '{query}': Official data confirms Q3 updates."]

    def run_crag(self, query: str, initial_docs: List[str]) -> List[str]:
        """Executes the CRAG corrective routing loop."""
        action_state = self.evaluate_retrieval(query, initial_docs)
        
        final_context = []
        if action_state == "correct":
            final_context = self.refine_knowledge_strip(initial_docs)
        elif action_state == "incorrect":
            final_context = self.trigger_web_fallback_search(query)
        elif action_state == "ambiguous":
            refined = self.refine_knowledge_strip(initial_docs)
            web_results = self.trigger_web_fallback_search(query)
            final_context = refined + web_results
            
        return final_context

# --- Execution Block ---
if __name__ == "__main__":
    crag = CorrectiveRAGPipeline()
    
    # Test Scenario: Vector store returns weak/missing data
    query = "What are the latest compliance updates for 2026?"
    weak_docs = ["error: document not found in vector index."]
    
    context = crag.run_crag(query, weak_docs)
    print("\n--- Final Corrected Context for Generator ---")
    for idx, c in enumerate(context, 1):
        print(f"{idx}. {c}")

## 3. Why CRAG is Essential for Enterprise Production
**Prevents Cascading Hallucinations:** Stopping bad retrieval data at the door prevents the generator LLM from weaving hallucinations out of irrelevant text chunks.

**Dynamic Self-Healing:** By fallback-switching to web searches or broader indexes when internal documents come up empty, your application stops returning dead-end "I cannot answer this" replies.